# Network Science (981G5) Assessment

Abstract\Introduction:
> Some text here

#### Contents
- [Heading 1]()
- [Heading 2]()

---

#### Notebook Setup

In [2]:
# ========================
# Installations
# ========================

# Install required dependencies
%pip install -q --upgrade pip
%pip install -q pandas statsbombpy networkx matplotlib jinja2 seaborn scipy

# Enable live auto-reloading for helpers.py updates
%load_ext autoreload
%autoreload 2

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [3]:
# ========================
# Imports
# ========================

import warnings
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
import seaborn as sns
from statsbombpy import sb
from statsbombpy.api_client import NoAuthWarning

# Custom helper module
import helper as hp

# Global notebook configurations
warnings.simplefilter("ignore", NoAuthWarning)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

### Other

## Statbomb API: Raw Data Import

This the section that works on importing the raw events data from the Statsbomb and processing it in the form which is suitable for network analysis. Events data refers to identifable on-field actions, i.e. pass, tackle, shot. 

---

Section Contents: 
- [Heading 1]()
- [Heading 2]()

---

#### 1.1 Competition Extraction
StatBomb structure their data is various tables and granularly layers. We want to work with the 2023/2024 FA Women's Super League (`competition_id: 37`, `season_id: 281`) data. The code below queries to the `competitions` table to confirm that we have the correct codes. 

In [4]:
# Retrieve the competitions
competitions_df = sb.competitions()
print(f"Total Competitions Returned: {len(competitions_df)}")

# Filter the DataFrame for WSL 2023/2024
target_filter = (competitions_df['competition_id'] == 37) & (competitions_df['season_id'] == 281)
wsl_competition = competitions_df[target_filter].to_dict(orient='records')[0]

# Print the extracted row
print("\nExtracted Competition Payload:")
print(wsl_competition)

Total Competitions Returned: 80

Extracted Competition Payload:
{'competition_id': 37, 'season_id': 281, 'country_name': 'England', 'competition_name': "FA Women's Super League", 'competition_gender': 'female', 'competition_youth': False, 'competition_international': False, 'season_name': '2023/2024', 'match_updated': '2026-04-11T13:05:10.794831', 'match_updated_360': nan, 'match_available_360': nan, 'match_available': '2026-04-11T13:05:10.794831'}


In [5]:
COMPETITION_ID = 37
SEASON_ID = 281

### 1.2 Match, Team, Player and Events Extractions
To construct a Passing Network (PassMap), we need to extract and aggregate event-level data across individual matches. Each match yields two networks for the competing teams. In order to compile these team-match network we need to compile the nesecary information to structure our network and call the API for the specific information. 

1. **Nodes (Players):** Extracted from match lineups to determine player identity, tactical position, and appearance duration.
2. **Edges (Passes):** Extracted from match events to capture successfully completed passes between teammates.
3. **Graph Attributes:** Extracted to store tactical context, such as starting formations.

The following code offload processing to utilities held in `helper.py`.

| Function | Primary Purpose | Output/Pipeline Role |
| :--- | :--- | :--- |
| `hp.fetch_match_details()` | Downloads event stream and lineup payloads in a single API call. | Captures match duration (`max_minute`) and raw payloads. |
| `hp.extract_team_roster()` &<br>`hp.extract_11_players()` | Parses substitution timestamps and calculates individual appearances. | Computes total minutes played and isolates the core 11 players with highest volume. |
| `hp.extract_team_formation()` | Queries tactical setups (e.g., `4-3-3`, `3-4-3`) from starting XI events. | Attaches tactical system metadata to the team network. |
| `hp.extract_successful_passes()` | Filters out incomplete or intercepted passes. | Returns completed pass actions needed to build adjacency matrices. |

The result is a list of match-team entries containing: match id, team name, starting formation, team total passes, the full player roster including subs and the main team of 11 players who played the most minutes. This gives us all of the information be needed to query the API, pull the pass data and construct a network.

In [42]:
# Extract full seasons worth of match_ids
matches_df = sb.matches(competition_id=COMPETITION_ID, season_id=SEASON_ID)
matches = matches_df["match_id"].to_list()[0:2]
print(f"WSL 23/24 contains {len(matches)} matches.")

match_records = []

for match in matches:
    events, lineups, max_minute = hp.fetch_match_details(match)
    for team_name, lineup_df in lineups.items():

        # Roster extraction 
        roster = hp.extract_team_roster(lineup_df, max_minute)
        top_11_players = hp.extract_11_players(roster)

        # Tactical and event extraction
        formation = hp.extract_team_formation(events, team_name)
        team_passes = hp.extract_successful_passes(events, team_name)
    
        match_records.append({
            "match_id": match,
            "team": team_name,
            "formation": formation,
            "total_passes": len(team_passes),
            "full_roster": roster,
            "top_11_players": top_11_players
        })

WSL 23/24 contains 2 matches.


In [46]:
print(f"{len(match_records)} team-match records extracted")
print(40*"=")
print("Sample[0]: ")
print("Match ID:", match_records[0]["match_id"])
print("Team Name:", match_records[0]["team"])
print("Formation:", match_records[0]["formation"])
print("Team Passes:", match_records[0]["total_passes"])
print("Full Roster", len(match_records[0]["full_roster"]))
print("Most Used Players:", len(match_records[0]["top_11_players"]))
print("Most Used Players:", match_records[0]["top_11_players"])
print("Top 3 Players in Roster:")
for p in match_records[0]["top_11_players"][:3]:
    print(
        f"  • {p['Player Name']} ({p['Position']}) - {p['Minutes Played']} mins"
    )


4 team-match records extracted
Sample[0]: 
Match ID: 3913082
Team Name: Brighton & Hove Albion WFC
Formation: 343
Team Passes: 204
Full Roster 16
Most Used Players: 11
Most Used Players: [{'Player Name': 'Maria Thorisdottir', 'Player ID': 4636, 'Position': 'Center Back', 'Starting Minute': 0, 'Ending Minute': 100, 'Minutes Played': 100}, {'Player Name': 'Sophie Baggaley', 'Player ID': 16376, 'Position': 'Goalkeeper', 'Starting Minute': 0, 'Ending Minute': 100, 'Minutes Played': 100}, {'Player Name': 'Emma Nanny Charlotte Kullberg', 'Player ID': 34500, 'Position': 'Left Wing Back', 'Starting Minute': 0, 'Ending Minute': 100, 'Minutes Played': 100}, {'Player Name': 'Elisabeth Terland', 'Player ID': 276301, 'Position': 'Center Forward', 'Starting Minute': 0, 'Ending Minute': 100, 'Minutes Played': 100}, {'Player Name': 'Guro Bergsvand', 'Player ID': 276304, 'Position': 'Right Center Back', 'Starting Minute': 0, 'Ending Minute': 100, 'Minutes Played': 100}, {'Player Name': 'Jorelyn Andrea 

### 1.3